In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.conf.set("fs.azure.account.key.saleessa.dfs.core.windows.net","access_key")

In [0]:
dim_product = spark.sql(
    '''select * from delta.`abfss://gold@saleessa.dfs.core.windows.net/dim_product`
    '''
)
dim_region = spark.sql(
    '''select * from delta.`abfss://gold@saleessa.dfs.core.windows.net/dim_region`
    '''
)
dim_sale_repres = spark.sql(
    '''select * from delta.`abfss://gold@saleessa.dfs.core.windows.net/dim_representative`
    '''
)
dim_date = spark.sql(
    '''select * from delta.`abfss://gold@saleessa.dfs.core.windows.net/dim_date`
    '''
)
dim_customer = spark.sql(
    '''select * from delta.`abfss://gold@saleessa.dfs.core.windows.net/dim_customer`
    '''
)


In [0]:
df_silver = spark.sql(
    '''select * from parquet.`abfss://silver@saleessa.dfs.core.windows.net/`
    '''
)

In [0]:
dim_fact = df_silver.join(dim_product, 'Product_Category').join(dim_region, 'Sales_Region').join(dim_sale_repres, 'sales_Representative').join(dim_date, 'Sale_Date').join(dim_customer, ['Customer_Age','Customer_Gender']).select(dim_product.product_key, dim_region.region_key, dim_sale_repres.representative_key, dim_date.date_key, dim_customer.customer_key,df_silver.Discount,df_silver.Net_Sales,df_silver.Sales_Amount,df_silver.Sales_ID)

In [0]:
from delta.tables import DeltaTable

In [0]:
path = "abfss://gold@saleessa.dfs.core.windows.net/dim_fact"

if DeltaTable.isDeltaTable(spark,path):
    deltatable  = DeltaTable.forPath(spark,path)
    deltatable.alias('target').mergea(dim_fact.alias('source'),'target.product_key == source.product_key AND target.region_key == source.region_key AND target.representative_key == source.representative_key AND target.date_key == source.date_key AND target.customer_key == source.customer_key').whenNotMatchedInsertAll().WhenMatchedUpdateAll().execute()
    print('file upserted')
else:
    dim_fact.write.format("delta").mode('overwrite').option('inferSchema','true').save(path)
    print('file created')


file created
